In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, ShortType, ArrayType

In [0]:
COLUMN_NAMES = {
    "id": "id",
    "followers": "follower_count",
    "genre": "genres",
    "name": "name",
    "popularity": "popularity_score"
}

df = (
    spark.table("spotify_project.bronze.spotify_artists")
    .dropDuplicates()
    .dropna(how="any", subset=["id", "name"])
)

str_columns = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
for c in str_columns:
    df = df.withColumn(c, F.trim(F.col(c)))

df = df.withColumn("followers", F.coalesce(F.col("followers"), F.lit(0))).filter(F.col("followers") > 0)

df = df.withColumn("genres", F.coalesce(F.from_json(F.col("genres"), ArrayType(StringType())), F.array().cast(ArrayType(StringType()))))

df = df.withColumn("popularity", F.when(F.col("popularity").rlike("^[0-9]+$"), F.col("popularity").cast(ShortType()))
                                  .when(F.col("popularity").isNull(), F.lit(0))
                  ).filter(F.col("popularity") > 0)

df.withColumnsRenamed(COLUMN_NAMES)

df.write.mode("overwrite").format("delta").saveAsTable("spotify_project.silver.artists_info")

In [0]:
%sql
SELECT * FROM spotify_project.silver.artists_info WHERE followers > 1000000